# Índice de Cobertura Curricular (IV-I)
`w(h)=λ·I(h)+(1-λ)·S(h)`, `m(a,h)`=coseno TF-IDF sumilla–habilidad, `c(k,h)=max_a m(a,h)·1[m≥τc]`, `ICC(k)=Σw·c/Σw`.

In [4]:
import re, ast, unicodedata
import numpy as np, pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

LAMBDA = 0.5
TAU_C = None   # None = calibrar contra la columna 'habilidades' de las sumillas; o fija un valor (p. ej. 0.10)
nrm = lambda s: unicodedata.normalize("NFKD", str(s).lower()).encode("ascii", "ignore").decode().strip()

# --- Peso de cada habilidad: w = λ I + (1-λ) S ---
W = pd.read_csv("skill_influencia.csv")[["habilidad", "I"]]
of = pd.read_excel("ofertas_limpias_skills.xlsx")
of["lista"] = of["skills"].fillna("").map(lambda s: [x.strip() for x in s.split(";") if x.strip()])
sal = of[of.sal_mid.notna()]
med = sal.sal_mid.median()
def prima(h):
    s = sal[sal.lista.map(lambda l: h in l)].sal_mid
    return s.median() / med - 1 if len(s) >= 3 else 0.0   # <3 ofertas con salario: prima neutra
W["prima"] = W.habilidad.map(prima)
W["S"] = (W.prima - W.prima.min()) / (W.prima.max() - W.prima.min())
W["w"] = LAMBDA * W.I + (1 - LAMBDA) * W.S
W.to_csv("skill_peso.csv", index=False)
W.sort_values("w", ascending=False).head(10)

,habilidad,I,prima,S,w
14,iso 9001,0.746269,0.984127,1.00,0.873134
20,iso 14001,0.708955,0.984127,1.00,0.854478
5,gestión de proyectos,0.820896,0.736111,0.80,0.810448
30,iso 45001,0.619403,0.984127,1.00,0.809701
31,liderazgo,0.611940,0.984127,1.00,0.805970
7,costos y presupuestos,0.805970,0.736111,0.80,0.802985
12,planeamiento,0.753731,0.736111,0.80,0.776866
9,api rest,0.783582,0.661706,0.74,0.761791
3,kpi,0.858209,0.488095,0.60,0.729104
28,sap,0.626866,0.736111,0.80,0.713433


In [5]:
# --- Correspondencia semántica m(a,h): coseno TF-IDF entre sumilla y habilidad ---
cur = pd.read_csv("sumillas_habilidades.csv").fillna({"descripcion": "", "habilidades": "[]", "curso": ""})
vec = TfidfVectorizer(strip_accents="unicode", ngram_range=(1, 2), sublinear_tf=True)
Xc = vec.fit_transform(cur.curso + ". " + cur.descripcion)
M = cosine_similarity(Xc, vec.transform(W.habilidad))          # cursos x habilidades

# --- Calibración de τc: se usa la columna 'habilidades' (inferida) como referencia en las
#     habilidades comunes a ambos catálogos. Sustituto del panel de coordinadores. ---
ref = cur.habilidades.map(lambda x: {nrm(h) for h in ast.literal_eval(x)})
comunes = [i for i, h in enumerate(W.habilidad) if any(nrm(h) in r for r in ref)]
Y = np.array([[nrm(W.habilidad[j]) in r for j in comunes] for r in ref])
def f1(t):
    P = M[:, comunes] >= t
    tp, fp, fn = (P & Y).sum(), (P & ~Y).sum(), (~P & Y).sum()
    return 2 * tp / (2 * tp + fp + fn) if tp else 0
grid = np.arange(0.02, 0.50, 0.01)
tau = TAU_C if TAU_C is not None else grid[int(np.argmax([f1(t) for t in grid]))]
print(f"Habilidades comunes: {len(comunes)} | τc = {tau:.2f} | F1 de calibración = {f1(tau):.3f}")

Habilidades comunes: 10 | τc = 0.04 | F1 de calibración = 0.906


In [6]:
# --- Cobertura c(k,h) e ICC(k) ---
C = np.where(M >= tau, M, 0)
w = W.w.values
carreras = {k: np.where(cur.carrera == k)[0] for k in cur.carrera.unique()}
cov = pd.DataFrame({k: C[i].max(axis=0) for k, i in carreras.items()}, index=W.habilidad)
icc = (cov.mul(w, axis=0).sum() / w.sum()).rename("ICC").sort_values(ascending=False)

# Aporte por asignatura y brechas (habilidades con peso y sin cobertura)
cur["aporte"] = (C * w).sum(axis=1) / w.sum()
brechas = pd.concat([W.assign(carrera=k)[cov[k].values == 0] for k in cov], ignore_index=True)
brechas = brechas.sort_values(["carrera", "w"], ascending=[True, False])[["carrera", "habilidad", "I", "S", "w"]]

icc.to_csv("icc_carrera.csv"); cov.to_csv("cobertura_skill_carrera.csv")
cur[["carrera", "curso", "aporte"]].to_csv("icc_curso.csv", index=False); brechas.to_csv("brechas.csv", index=False)
print(icc.round(3)); brechas.groupby("carrera").head(5)

Ingeniería Industrial     0.052
Ingeniería Informática    0.049
Ingeniería Mecatrónica    0.046
Ingeniería Electrónica    0.033
Name: ICC, dtype: float64


,carrera,habilidad,I,S,w
8,Ingeniería Electrónica,iso 9001,0.746269,1.00,0.873134
5,Ingeniería Electrónica,costos y presupuestos,0.805970,0.80,0.802985
7,Ingeniería Electrónica,planeamiento,0.753731,0.80,0.776866
6,Ingeniería Electrónica,api rest,0.783582,0.74,0.761791
2,Ingeniería Electrónica,kpi,0.858209,0.60,0.729104
56,Ingeniería Industrial,api rest,0.783582,0.74,0.761791
53,Ingeniería Industrial,kpi,0.858209,0.60,0.729104
66,Ingeniería Industrial,sap,0.626866,0.80,0.713433
55,Ingeniería Industrial,agile,0.813433,0.60,0.706716
52,Ingeniería Industrial,power bi,0.865672,0.40,0.632836
